### Developing NGS analysis pipeline for amplicon-base FCoV sequencing
 - First, prepare index file that covers all available FCoV genomoes

In [1]:
import pandas as pd
from Bio import SeqIO, Entrez
from Bio.Seq import Seq 
import matplotlib as mpl
from matplotlib.font_manager import FontProperties
mpl.rcParams['svg.fonttype'] = 'none'
# Path to your Arial.ttf file
arial_font_path = '/Users/skawakubo/python/fonts/Arial.ttf'
# Create a FontProperties object for the Arial font
arial_font = FontProperties(fname=arial_font_path)
mpl.rcParams['font.family'] = arial_font.get_name()
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
import subprocess

Entrez.email = 'skawakubo@g.ecc.u-tokyo.ac.jp'

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
def fasta_to_dataframe(fasta_file):
    ids = []
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        ids.append(record.id)
        sequences.append(str(record.seq))

    df = pd.DataFrame({
        "ID": ids,
        "seq": sequences
    })
    return df

In [10]:
FCoV_seqs = fasta_to_dataframe('FCoV_full_NCBI_Izumi_250818.fasta')
FCoV_seqs['N_count'] = FCoV_seqs['seq'].apply(lambda x: x.count('n'))
FCoV_seqs

,ID,seq,N_count
0,NC_002306.3,acttttaaagtaaagtgagtgtagcgtggctataactcttctttta...,0
1,AY994055.1,acttttaaagtaaagtgagtgtagcgtggctataactcttctttta...,0
2,PP526173.1,acttttaaagtaaagtgagtgtagcgtggctataactcttctttta...,0
3,OQ311323.1,acttttaaagtaaagtgagtgtagcgtggctataactcttctttta...,0
4,KC461235.1,acttttaaagtaaagtgagtgtagcgtggctataactcttctttta...,0
...,...,...,...
157,775024S_S3_L001,gacaccaactcgaactaaacgaaatatttgtctctctatgaaacca...,0
158,771819S_S4_L001,cggacaccaactcgaactaaacgaaatatttgtctctctatgaaac...,0
159,PP254212.1,tgtggcgtggctataactctgcttgkngtttaacctgccttgtgct...,4633
160,PP254214.1,tttactttaacctgccttgtgctagatttgtcttcggacaccaact...,4492


In [11]:
FCoV_meta = pd.read_csv('FCoV_all_full_meta_250830_modified.tsv', sep='\t')
FCoV_meta

,Accession,Isolate,Collection_Date,Country,Host,Tissue_Specimen_Source,Length,Source,Note,FECV_or_FIPV,Unnamed: 10
0,NC_002306.3,79-1146,NaN,USA,NaN,NaN,29355.0,NCBI,NaN,FIPV,NaN
1,PP254206.1,ZD/22/332i,2022/7/17,India,Felis catus,feces,29229.0,NCBI,NaN,FECV,NaN
2,PP254207.1,ZD/22/337i,2022/7/17,India,Felis catus,feces,29229.0,NCBI,NaN,FECV,NaN
3,PP254208.1,ZD/22/385i,2022/7/19,India,Felis catus,feces,29286.0,NCBI,NaN,FECV,NaN
4,PP254209.1,ZD/22/519i,2022/7/21,India,Felis catus,feces,29221.0,NCBI,NaN,FECV,NaN
...,...,...,...,...,...,...,...,...,...,...,...
157,FECV43,FECV43,2025/6/16,Japan,Felis catus,feces,NaN,Izumi,NaN,FECV,NaN
158,FECV46,FECV46,2025/7/7,Japan,Felis catus,feces,NaN,Izumi,NaN,FECV,NaN
159,FECV47,FECV47,2025/7/7,Japan,Felis catus,feces,NaN,Izumi,NaN,FECV,NaN
160,FECV51,FECV51,2025/7/7,Japan,Felis catus,feces,NaN,Izumi,NaN,FECV,NaN


In [12]:
df_FCoV = pd.merge(FCoV_meta, FCoV_seqs, left_on='Accession', right_on='ID')
df_FCoV.drop(columns=['ID'], inplace=True)
df_FCoV['Length'] = df_FCoV['seq'].apply(len)
df_FCoV

,Accession,Isolate,Collection_Date,Country,Host,Tissue_Specimen_Source,Length,Source,Note,FECV_or_FIPV,Unnamed: 10,seq,N_count
0,NC_002306.3,79-1146,NaN,USA,NaN,NaN,29353,NCBI,NaN,FIPV,NaN,acttttaaagtaaagtgagtgtagcgtggctataactcttctttta...,0
1,PP254206.1,ZD/22/332i,2022/7/17,India,Felis catus,feces,29229,NCBI,NaN,FECV,NaN,tctgcttttgctttaacctgccttgtgctagatttgtcttcggaca...,3853
2,PP254207.1,ZD/22/337i,2022/7/17,India,Felis catus,feces,29229,NCBI,NaN,FECV,NaN,aactctgcttggggtttaacctgccttgtgctagatttgtcttcgg...,12
3,PP254208.1,ZD/22/385i,2022/7/19,India,Felis catus,feces,29286,NCBI,NaN,FECV,NaN,tcttcatttaktttaacctgccttgtgctagatttgtcttcggaca...,3000
4,PP254209.1,ZD/22/519i,2022/7/21,India,Felis catus,feces,29221,NCBI,NaN,FECV,NaN,tcttcgtttgctttaacctgccttgtgctagatttgtcttcggaca...,2610
...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,FECV43,FECV43,2025/6/16,Japan,Felis catus,feces,25587,Izumi,NaN,FECV,NaN,ggtataccattttccacctttaaaacaccagctagcagttttattg...,0
158,FECV46,FECV46,2025/7/7,Japan,Felis catus,feces,27808,Izumi,NaN,FECV,NaN,gaccgcgacattgcaagaaatttcgtgaatggaacaatctggaaag...,0
159,FECV47,FECV47,2025/7/7,Japan,Felis catus,feces,29113,Izumi,NaN,FECV,NaN,gacttttaaagtaaagtgagtgtagcgtggctataactcttctttt...,0
160,FECV51,FECV51,2025/7/7,Japan,Felis catus,feces,29378,Izumi,NaN,FECV,NaN,cacttttaaagtaaagtgagtgtagcgtggctataactcttctttt...,0


In [17]:
df_FCoV_for_mappingidx = df_FCoV[df_FCoV['N_count'] == 0]
df_FCoV_for_mappingidx

,Accession,Isolate,Collection_Date,Country,Host,Tissue_Specimen_Source,Length,Source,Note,FECV_or_FIPV,Unnamed: 10,seq,N_count
0,NC_002306.3,79-1146,NaN,USA,NaN,NaN,29353,NCBI,NaN,FIPV,NaN,acttttaaagtaaagtgagtgtagcgtggctataactcttctttta...,0
11,PP908788.1,FCoV/Thailand/CU27528/2021,21-Jul,Thailand,Felis catus,feces,28716,NCBI,NaN,FECV,NaN,tgagttccaaacaatttaagatcctcgttaatgaggactaccaagt...,0
12,PP908789.1,FCoV/Thailand/CU27724/2021,21-Aug,Thailand,Felis catus,feces,29055,NCBI,NaN,FECV,NaN,gagagaacggtgcgccagggttccgtccctgtttggtaagtcactt...,0
14,PP854701.1,OM1164,8-Nov,USA,Otocolobus manul,lung,29180,NCBI,NaN,FIPV,NaN,cggacaccaactcgaactaaacgaaatatttgtctctctatgaagc...,0
15,PP854702.1,DCB091,8-Nov,USA,Felis catus,lymph,29180,NCBI,NaN,FIPV,NaN,cggacaccaactcgaactaaacgaaatatttgtctctctatgaagc...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,FECV43,FECV43,2025/6/16,Japan,Felis catus,feces,25587,Izumi,NaN,FECV,NaN,ggtataccattttccacctttaaaacaccagctagcagttttattg...,0
158,FECV46,FECV46,2025/7/7,Japan,Felis catus,feces,27808,Izumi,NaN,FECV,NaN,gaccgcgacattgcaagaaatttcgtgaatggaacaatctggaaag...,0
159,FECV47,FECV47,2025/7/7,Japan,Felis catus,feces,29113,Izumi,NaN,FECV,NaN,gacttttaaagtaaagtgagtgtagcgtggctataactcttctttt...,0
160,FECV51,FECV51,2025/7/7,Japan,Felis catus,feces,29378,Izumi,NaN,FECV,NaN,cacttttaaagtaaagtgagtgtagcgtggctataactcttctttt...,0


In [18]:
with open('FCoV_full_for_mappingidx.fasta', 'w') as f:
    for index, row in df_FCoV_for_mappingidx.iterrows():
        f.write(f">{row['Accession']}\n{row['seq']}\n")

#### Exported 114 FCoV wg seqs to be used for mapping NGS reads